In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
print("✅ Libraries loaded!")

✅ Libraries loaded!


In [2]:

df = pd.read_csv(r"C:\Users\hp\Downloads\ML\spam.csv")
df.head()


,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
df.shape

(5572, 2)

In [4]:
df['Category'].value_counts()

Category
ham     4825
spam     747
Name: count, dtype: int64

In [5]:
ham = df[df.Category == 'ham']
spam = df[df.Category == 'spam']

In [6]:
ham_sample = ham.sample(n=747)    
new_dataset = pd.concat([ham_sample,spam],axis=0)

In [7]:
new_dataset['Category'].value_counts()

Category
ham     747
spam    747
Name: count, dtype: int64

In [8]:
def preprocess_text(text):

    words = word_tokenize(text)    
    words = [word.lower() for word in words if word.isalpha()]   
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]       
    stemmer = PorterStemmer()
    words = [stemmer.stem(word) for word in words]
    return " ".join(words)

In [9]:
new_dataset["processed_message"] = new_dataset["Message"].apply(preprocess_text)

In [10]:
new_dataset.head()

,Category,Message,processed_message
522,ham,Shall i come to get pickle,shall come get pickl
3746,ham,"Aight, let me know when you're gonna be around...",aight let know gon na around usf
1491,ham,Cant believe i said so many things to you this...,cant believ said mani thing morn realli want s...
1725,ham,There bold 2 &lt;#&gt; . Is that yours,bold lt gt
3117,ham,Uncle Abbey! Happy New Year. Abiola,uncl abbey happi new year abiola


In [11]:
x = new_dataset['processed_message']
y = new_dataset['Category']

x_train, x_test, y_train, y_test = train_test_split(x, y,
    test_size=0.2,
    random_state=42
)

print(f"Training data size: {len(x_train)}")
print(f"Test data size: {len(x_test)}")

vectorizer = TfidfVectorizer(max_features=5000)
x_train_tf = vectorizer.fit_transform(x_train)
x_test_tf = vectorizer.transform(x_test)

Training data size: 1195
Test data size: 299


In [12]:
model = MultinomialNB()
model.fit(x_train_tf, y_train)

print("✅ Model trained successfully!")

✅ Model trained successfully!


In [13]:
y_pred = model.predict(x_test_tf)
accuracy = accuracy_score(y_test, y_pred)

print("=" * 55)
print("   SPAM CLASSIFICATION RESULTS")
print("=" * 55)
print(f"\n🎯 Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
print(f"\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("--- Confusion Matrix ---")
print(f"                Predicted Ham  Predicted Spam")
print(f"Actual Ham      {cm[0][0]:>10}    {cm[0][1]:>10}")
print(f"Actual Spam     {cm[1][0]:>10}    {cm[1][1]:>10}")

   SPAM CLASSIFICATION RESULTS

🎯 Accuracy: 0.9565 (95.7%)

--- Detailed Classification Report ---
              precision    recall  f1-score   support

         ham       0.95      0.96      0.96       145
        spam       0.96      0.95      0.96       154

    accuracy                           0.96       299
   macro avg       0.96      0.96      0.96       299
weighted avg       0.96      0.96      0.96       299

--- Confusion Matrix ---
                Predicted Ham  Predicted Spam
Actual Ham             139             6
Actual Spam              7           147


In [14]:
def predict_email(text):
    stop_words = set(stopwords.words('english'))
    cleaned = ' '.join(word.lower() for word in word_tokenize(text) if word.isalpha())
    cleaned = ' '.join(word for word in cleaned.split() if word not in stop_words)

    features = vectorizer.transform([cleaned])
    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0]

    emoji = "✅" if prediction == "ham" else "🚫"
    print(f"{emoji} Prediction: {prediction.upper()}")
    print(f"   Confidence — Ham: {probability[0]:.2%}, Spam: {probability[1]:.2%}")
    return prediction

In [16]:
print("--- Testing Custom Emails ---\n")

print("Email 1:")
print("Hey, are we still meeting for lunch tomorrow?")
predict_email("Hey, are we still meeting for lunch tomorrow?")

print("\nEmail 2:")
print("CONGRATULATIONS! You won a FREE iPhone! Click NOW to claim!")
predict_email("CONGRATULATIONS! You won a FREE iPhone! Click NOW to claim!")

print("\nEmail 3:")
print("Please find the quarterly report attached as discussed.")
predict_email("Please find the quarterly report attached as discussed.")

--- Testing Custom Emails ---

Email 1:
Hey, are we still meeting for lunch tomorrow?
✅ Prediction: HAM
   Confidence — Ham: 87.02%, Spam: 12.98%

Email 2:
CONGRATULATIONS! You won a FREE iPhone! Click NOW to claim!
🚫 Prediction: SPAM
   Confidence — Ham: 6.78%, Spam: 93.22%

Email 3:
Please find the quarterly report attached as discussed.
🚫 Prediction: SPAM
   Confidence — Ham: 30.66%, Spam: 69.34%


np.str_('spam')